# Stacking-Ensemble

In diesem Notebook wird untersucht, ob sich die Vorhersageleistung durch die
Kombination unterschiedlicher Modelltypen weiter verbessern lässt. Als
Basismodelle werden eine logistische Regression, ein optimierter Random Forest
und das zuvor ausgewählte XGBoost-Modell verwendet.

Die Bewertung erfolgt mittels äußerer geschichteter 5-Fold-Cross-Validation.
Innerhalb jedes Trainingsfolds erzeugt der StackingClassifier mittels einer
separaten inneren Cross-Validation die Vorhersagen für das Meta-Modell.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression , LogisticRegressionCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    cross_val_predict
)

from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

from xgboost import XGBClassifier

In [2]:
project_root = Path.cwd()
processed_dir = project_root / "data" / "processed"

In [3]:
train_data = pd.read_parquet(
    processed_dir / "train.parquet"
)

test_data = pd.read_parquet(
    processed_dir / "test.parquet"
)

print("Trainingsdaten:", train_data.shape)
print("Testdaten:", test_data.shape)

Trainingsdaten: (160000, 201)
Testdaten: (40000, 201)


In [4]:
X_train = train_data.drop(columns=["target"])
y_train = train_data["target"]

X_test = test_data.drop(columns=["target"])
y_test = test_data["target"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (160000, 200)
y_train: (160000,)
X_test: (40000, 200)
y_test: (40000,)


In [5]:
scoring = {
    "PR_AUC": "average_precision",
    "ROC_AUC": "roc_auc",

    "Precision": make_scorer(
        precision_score,
        zero_division=0
    ),

    "Recall": make_scorer(
        recall_score,
        zero_division=0
    ),

    "F1": make_scorer(
        f1_score,
        zero_division=0
    )
}

## Basismodelle

Die Basismodelle wurden aus den vorherigen Modelluntersuchungen übernommen.
Durch die Kombination eines linearen Modells, eines Bagging-Verfahrens und
eines Boosting-Verfahrens soll geprüft werden, ob komplementäre
Vorhersagestrukturen genutzt werden können.

In [6]:
logreg_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            C=0.001,
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

In [7]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    criterion="gini",
    max_depth=40,
    min_samples_split=2,
    min_samples_leaf=10,
    max_features="log2",
    bootstrap=True,
    class_weight=None,
    max_samples=0.8,
    random_state=42,
    n_jobs=4
)

In [8]:
xgboost_model = XGBClassifier(
    objective="binary:logistic",
    tree_method="hist",
    eval_metric="logloss",

    n_estimators=1500,
    learning_rate=0.10,

    max_depth=2,
    min_child_weight=3,

    subsample=0.8,
    colsample_bytree=0.7,

    scale_pos_weight=1.0,
    gamma=0,

    reg_alpha=0.01,
    reg_lambda=1,

    random_state=42,
    n_jobs=4
)

In [9]:
inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=43
)

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [10]:
base_estimators = [
    ("logreg", logreg_model),
    ("random_forest", random_forest_model),
    ("xgboost", xgboost_model)
]

## Meta-Modell

Als Meta-Modell wird eine logistische Regression verwendet. Der
Regularisierungsparameter C wird innerhalb jedes äußeren Trainingsfolds
erneut durch Cross-Validation bestimmt. Dadurch wird vermieden, den
Validierungsfold für die Auswahl des Meta-Modells zu verwenden.

In [11]:
meta_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=44
)

meta_model = LogisticRegressionCV(
    Cs=[0.01, 0.1, 1, 10],
    cv=meta_cv,
    scoring="average_precision",
    max_iter=1000,
    n_jobs=1,
    random_state=42
)


In [12]:
stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_model,

    cv=inner_cv,
    stack_method="predict_proba",

    passthrough=False,
    n_jobs=1
)

In [13]:
stacking_cv_results = cross_validate(
    stacking_model,
    X_train,
    y_train,
    cv=outer_cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=True,
    return_estimator=True,
    verbose=2
)

/home/lenovo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/lenovo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


[CV] END .................................................... total time=15.3min


/home/lenovo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/lenovo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


[CV] END .................................................... total time=11.8min


/home/lenovo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/lenovo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


[CV] END .................................................... total time= 9.8min


/home/lenovo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/lenovo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


[CV] END .................................................... total time= 9.7min


/home/lenovo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/lenovo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


[CV] END .................................................... total time= 9.8min


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed: 56.8min finished


In [14]:
for fold, estimator in enumerate(
    stacking_cv_results["estimator"],
    start=1
):
    selected_c = estimator.final_estimator_.C_[0]

    print(
        f"Fold {fold}: "
        f"ausgewähltes C = {selected_c}"
    )

Fold 1: ausgewähltes C = 1.0
Fold 2: ausgewähltes C = 0.1
Fold 3: ausgewähltes C = 1.0
Fold 4: ausgewähltes C = 1.0
Fold 5: ausgewähltes C = 1.0


In [15]:
for metric in scoring:
    train_scores = stacking_cv_results[f"train_{metric}"]
    validation_scores = stacking_cv_results[f"test_{metric}"]

    print(f"{metric}:")

    print(
        f"  Training:    {train_scores.mean():.4f} "
        f"(± {train_scores.std():.4f})"
    )

    print(
        f"  Validierung: {validation_scores.mean():.4f} "
        f"(± {validation_scores.std():.4f})"
    )

print()

print(
    "Mittlere Trainingszeit pro Fold:",
    round(stacking_cv_results["fit_time"].mean(), 2),
    "Sekunden"
)

PR_AUC:
  Training:    0.8342 (± 0.0289)
  Validierung: 0.6027 (± 0.0096)
ROC_AUC:
  Training:    0.9674 (± 0.0089)
  Validierung: 0.8916 (± 0.0028)
Precision:
  Training:    0.8648 (± 0.0102)
  Validierung: 0.7217 (± 0.0151)
Recall:
  Training:    0.6074 (± 0.0359)
  Validierung: 0.3978 (± 0.0111)
F1:
  Training:    0.7132 (± 0.0288)
  Validierung: 0.5127 (± 0.0098)

Mittlere Trainingszeit pro Fold: 676.36 Sekunden


In [16]:
meta_details = []

model_names = [
    name for name, _ in base_estimators
]

for fold, estimator in enumerate(
    stacking_cv_results["estimator"],
    start=1
):
    meta = estimator.final_estimator_

    row = {
        "Fold": fold,
        "Selected_C": meta.C_[0]
    }

    for model_name, coefficient in zip(
        model_names,
        meta.coef_[0]
    ):
        row[model_name] = coefficient

    meta_details.append(row)

meta_details = pd.DataFrame(meta_details)

meta_details.round(4)

,Fold,Selected_C,logreg,random_forest,xgboost
0,1,1.0,2.3287,7.8538,4.2617
1,2,0.1,2.4559,3.8470,4.6358
2,3,1.0,2.3210,8.0791,4.1701
3,4,1.0,2.3511,7.9875,4.2136
4,5,1.0,2.3333,7.7436,4.2570


In [17]:
results_dir = project_root / "results" / "cv"

xgb_cv_results = pd.read_csv(
    results_dir / "xgboost_finalists_cv.csv"
)

xgb_scores = (
    xgb_cv_results
    .loc[
        xgb_cv_results["Modell"] == "XGBoost_1500"
    ]
    .sort_values("Fold")["PR_AUC"]
    .to_numpy()
)

stack_scores = stacking_cv_results["test_PR_AUC"]

fold_comparison = pd.DataFrame({
    "Fold": range(1, len(stack_scores) + 1),
    "XGBoost": xgb_scores,
    "Stacking": stack_scores
})

fold_comparison["Differenz_Stacking_minus_XGB"] = (
    fold_comparison["Stacking"]
    - fold_comparison["XGBoost"]
)

fold_comparison.round(4)

,Fold,XGBoost,Stacking,Differenz_Stacking_minus_XGB
0,1,0.6020,0.5995,-0.0024
1,2,0.6006,0.5980,-0.0026
2,3,0.6239,0.6199,-0.0039
3,4,0.5926,0.5916,-0.0010
4,5,0.6057,0.6046,-0.0011


## Ergebnis

Das Stacking erreicht eine mittlere PR-AUC von 0,603 und eine ROC-AUC von
0,892. Damit erzielt es eine hohe Vorhersageleistung, liegt bei der PR-AUC
jedoch in allen fünf Folds geringfügig unter XGBoost_1500.

Da das Stacking zusätzlich eine deutlich höhere Trainingszeit verursacht,
ergibt sich kein ausreichender Vorteil gegenüber dem stärksten Einzelmodell.
XGBoost wird deshalb als finales Modell beibehalten.

In [18]:
final_stacking_model = stacking_model

final_stacking_model.fit(
    X_train,
    y_train
)

stacking_test_proba = final_stacking_model.predict_proba(
    X_test
)[:, 1]

/home/lenovo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/lenovo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


In [19]:
results_dir = project_root / "results" / "cv"
results_dir.mkdir(parents=True, exist_ok=True)

stacking_scores = pd.DataFrame({
    "Modell": ["Stacking"] * 5,
    "Fold": range(1, 6),
    "PR_AUC": stacking_cv_results["test_PR_AUC"],
    "ROC_AUC": stacking_cv_results["test_ROC_AUC"],
    "Precision": stacking_cv_results["test_Precision"],
    "Recall": stacking_cv_results["test_Recall"],
    "F1": stacking_cv_results["test_F1"],
    "Fit_Time": stacking_cv_results["fit_time"],
    "Schwellenwert": 0.5
})

stacking_scores.to_csv(
    results_dir / "stacking_cv.csv",
    index=False
)